# Christmas at Wilderness Outfitters: the stock plan

Three days into Christmas trading, Martha Thompson, Wilderness's Chief Customer Officer, has three questions for the merchandising team: which tents go back to the supplier, which tents move between stores, and how many to reorder by 13 November.

This notebook answers them from the raw data, in the same order as Lab 2, and goes further on the faulty tents: what reviewers say about them, which customers have already called a store, and a brief for Kestrel, the supplier. Every step is SQL you can read and run again, with a chart where a picture says it faster.

**Today is Wednesday 4 November 2026.** The data runs to Tuesday 3 November. It is synthetic training data.

| Part | Question | Snowflake feature | What it creates |
|---|---|---|---|
| 1 | Does each state have enough tents? | SQL | Nothing; it reads the data |
| 2 | Why are Summit Tents coming back? | `AI_SENTIMENT`, `AI_COUNT_TOKENS`, `AI_CLASSIFY`, `AI_AGG` | `REVIEW_SENTIMENT`, `TENT_RETURN_REASONS` |
| 3 | Who has already called about it? | `AI_FILTER` | `TENT_CALLS` |
| 4 | Who gets a sleeping-bag offer? | `AI_COUNT_TOKENS`, `AI_COMPLETE`, `AI_FILTER` | `SLEEPING_BAG_OFFER`, with five emails |
| 5 | What does each store do? | Dynamic table | `CHRISTMAS_PLAN` |

**How to run it.** Select **Run all**, or run the cells one at a time from the top. It takes about five minutes. Each AI function runs inside Snowflake, next to the data, and is charged by the amount of text it reads and writes, so the notebook counts the tokens before its two largest runs. Running it again replaces the tables it built, including any you built with Cortex Code in the lab.

## Before you start

Point the session at the workshop database, warehouse and schema. Everything this notebook creates goes in `LAB2_ANALYTICS`.

In [ ]:
%%sql -r context
-- Point the session at the workshop
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WILDERNESS_WH;
USE DATABASE WILDERNESS_HOL;
USE SCHEMA LAB2_ANALYTICS;

The charts share one look: Wilderness green for tents to spare, red for trouble. This cell sets it up and adds a helper that turns a SQL result into a small pandas table for plotting.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

GREEN, RED, AMBER, GREY, INK = "#2E7D46", "#C0392B", "#C77C1E", "#A7B1AB", "#1C2B22"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": GREY, "axes.labelcolor": INK,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": INK, "ytick.color": INK,
})


def as_pandas(result, numeric=()):
    """SQL cell results are Snowpark DataFrames on Notebook Runtime 2.6 and later, pandas before that."""
    df = result.copy() if isinstance(result, pd.DataFrame) else result.to_pandas()
    for column in numeric:
        df[column] = df[column].astype(float)
    return df


def titled(ax, title, note):
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold", color=INK, pad=22)
    ax.text(0, 1.02, note, transform=ax.transAxes, fontsize=9, color="#55625A")

## 1. Does each state have enough tents?

Wilderness bet on tents everywhere: it bought 40% more tents for every store in every state. Before anyone moves a tent, check the bet against the raw data, state by state.

### Stock against last Christmas

Each state's tents on hand at 3 November, against the tents it sold over the same weeks last year, 4 November to 24 December.

In [ ]:
%%sql -r tent_stock
WITH tents AS (SELECT PRODUCT_ID FROM WILDERNESS_HOL.RAW.DIM_PRODUCT WHERE CATEGORY = 'Tents & Shelters'),
on_hand AS (
  SELECT s.STATE, SUM(i.ON_HAND_UNITS) AS TENTS_ON_HAND
  FROM WILDERNESS_HOL.RAW.FCT_INVENTORY i
  JOIN tents t ON t.PRODUCT_ID = i.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = i.STORE_ID
  GROUP BY 1),
sold AS (
  SELECT s.STATE, SUM(f.QUANTITY) AS TENTS_SOLD_LY
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-04' AND '2025-12-24'
  GROUP BY 1)
SELECT o.STATE, o.TENTS_ON_HAND, s.TENTS_SOLD_LY, ROUND(o.TENTS_ON_HAND / s.TENTS_SOLD_LY, 2) AS STOCK_VS_LAST_YEAR
FROM on_hand o JOIN sold s ON s.STATE = o.STATE
ORDER BY 1;

**What you should see:** `STOCK_VS_LAST_YEAR` is about 1.4 in every state. The extra tents went everywhere.

### Demand so far

The first three days of Christmas trading, 1 to 3 November, against the same three days last year.

In [ ]:
%%sql -r tent_growth
SELECT s.STATE,
       SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)) AS TENTS_2025,
       SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0)) AS TENTS_2026,
       ROUND(TENTS_2026 / TENTS_2025 - 1, 3) AS GROWTH
FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
WHERE p.CATEGORY = 'Tents & Shelters'
  AND (f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03')
GROUP BY 1
ORDER BY GROWTH DESC;

In [ ]:
growth = as_pandas(tent_growth, numeric=["GROWTH"]).sort_values("GROWTH")
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.barh(growth["STATE"], growth["GROWTH"] * 100,
               color=[RED if g > 0.4 else GREEN for g in growth["GROWTH"]])
ax.bar_label(bars, labels=[f"+{g:.0%}" for g in growth["GROWTH"]], padding=4, color=INK)
ax.axvline(40, color=INK, linestyle="--", linewidth=1)
ax.text(41, -0.6, "40% more stock", fontsize=8, color=INK)
ax.set_xlabel("Tent sales, 1 to 3 Nov 2026 against 2025 (%)")
titled(ax, "Demand is up most in New South Wales and Queensland",
       "Red: demand has grown by more than the 40% of extra stock")
plt.show()

**What you should see:** two groups. New South Wales and Queensland are up about 60%; every other state is up about 10%. So those two states, with 1.4 times the stock against 1.6 times the demand, run out before Christmas Eve, and the rest have tents left over.

## 2. Why are Summit Tents coming back?

Some Summit Tents are coming back broken. Kestrel, the supplier, takes back faulty stock, so Wilderness needs to know which tents are faulty and be able to show why. This part reads what customers wrote, four ways.

### The return rate by batch

How many Summit Tents from each production batch were sold, and how many came back.

In [ ]:
%%sql -r return_rate
SELECT f.BATCH_CODE, SUM(f.QUANTITY) AS SOLD, SUM(COALESCE(r.RETURN_QUANTITY, 0)) AS RETURNED,
       ROUND(RETURNED / SOLD, 3) AS RETURN_RATE
FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
LEFT JOIN WILDERNESS_HOL.RAW.FCT_RETURN r ON r.SALE_LINE_ID = f.SALE_LINE_ID
WHERE f.PRODUCT_ID = 'SKU-TNT-0102' AND f.BATCH_CODE IS NOT NULL
GROUP BY 1
ORDER BY RETURN_RATE DESC;

In [ ]:
rates = as_pandas(return_rate, numeric=["RETURN_RATE"]).sort_values("RETURN_RATE")
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.barh(rates["BATCH_CODE"], rates["RETURN_RATE"] * 100,
               color=[RED if b == "B26-07" else GREY for b in rates["BATCH_CODE"]])
ax.bar_label(bars, labels=[f"{r:.0%}" for r in rates["RETURN_RATE"]], padding=4, color=INK)
ax.set_xlabel("Summit Tents returned, as a share of those sold (%)")
titled(ax, "One batch comes back four times as often", "Summit Tent, every batch on record")
plt.show()

**What you should see:** batch `B26-07` comes back at about 18%; every other batch at about 4%. `B26-07` reached stores on 1 July.

### What reviewers say: AI_SENTIMENT

A return rate says that something is wrong, not what. The product reviews say what. `AI_SENTIMENT` reads each Summit Tent review and scores it overall and for each part of the tent you name, here the poles, the seams, the setup, the size and the value, as positive, negative, neutral or mixed. It saves the scores in a table, so the charts below read the table rather than asking the model again.

In [ ]:
%%sql -r review_sentiment
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.REVIEW_SENTIMENT AS
SELECT FEEDBACK_ID, FEEDBACK_DATE, RATING,
       IFF(FEEDBACK_DATE >= '2026-07-01', 'Since 1 July', 'Before 1 July') AS PERIOD,
       AI_SENTIMENT(FEEDBACK_TEXT, ['poles', 'seams', 'setup', 'size', 'value']) AS SENTIMENT
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE PRODUCT_ID = 'SKU-TNT-0102'
  AND FEEDBACK_SOURCE = 'PRODUCT_REVIEW';

Each score is a small JSON object. `LATERAL FLATTEN` turns it into one row per part of the tent, so the reviews before and after 1 July can be counted side by side.

In [ ]:
%%sql -r sentiment_by_aspect
SELECT PERIOD, c.value:name::VARCHAR AS ASPECT,
       COUNT(*) AS REVIEWS,
       COUNT_IF(c.value:sentiment::VARCHAR = 'negative') AS NEGATIVE,
       ROUND(NEGATIVE / REVIEWS, 3) AS NEGATIVE_SHARE
FROM WILDERNESS_HOL.LAB2_ANALYTICS.REVIEW_SENTIMENT,
     LATERAL FLATTEN(SENTIMENT:categories) c
GROUP BY 1, 2
ORDER BY 2, 1;

In [ ]:
aspects = as_pandas(sentiment_by_aspect, numeric=["NEGATIVE_SHARE"])
order = ["value", "size", "setup", "seams", "poles", "overall"]
table = aspects.pivot(index="ASPECT", columns="PERIOD", values="NEGATIVE_SHARE").reindex(order) * 100
fig, ax = plt.subplots(figsize=(8, 4))
y = range(len(table))
ax.barh([i + 0.2 for i in y], table["Before 1 July"], height=0.4, color=GREY, label="Before 1 July")
ax.barh([i - 0.2 for i in y], table["Since 1 July"], height=0.4, color=RED, label="Since 1 July")
ax.set_yticks(list(y), table.index)
ax.set_xlabel("Summit Tent reviews negative about it (%)")
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.16))
titled(ax, "Since B26-07 arrived, reviewers turned on the poles", "Share of all Summit Tent reviews, by part of the tent")
plt.show()

**What you should see:** before 1 July, about one review in five is negative and hardly any complain about the poles. Since 1 July, about two in five are negative and more than one in three complains about the poles. Every other part of the tent scores worse too: once a pole snaps or a seam leaks, reviewers mark down the setup, the size and the value as well.

### Count the cost first: AI_COUNT_TOKENS

The next cell labels 500 return comments. AI functions are charged by tokens, roughly three quarters of a word each, so check the size of a run before you start it. `AI_COUNT_TOKENS` counts the input tokens for a function without calling the model, and costs only the warehouse time.

In [ ]:
%%sql -r classify_cost
SELECT COUNT(*) AS COMMENTS,
       SUM(AI_COUNT_TOKENS('ai_classify', FEEDBACK_TEXT,
           ['POLE_FAULT', 'SEAM_FAULT', 'COULD_NOT_PITCH', 'CHANGE_OF_MIND', 'OTHER'])) AS INPUT_TOKENS,
       ROUND(INPUT_TOKENS / COMMENTS) AS TOKENS_PER_COMMENT
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE PRODUCT_ID = 'SKU-TNT-0102'
  AND FEEDBACK_SOURCE = 'RETURN_COMMENT'
  AND FEEDBACK_DATE >= '2026-07-01';

**What you should see:** 500 comments and about 116,000 input tokens, about 230 a comment. Most of that is the labels and instructions sent with each comment, not the comment itself.

### Label every return comment: AI_CLASSIFY

The returns desk codes nearly every return as faulty, so the codes cannot say which returns are real faults. The customer's comment can: a snapped pole or a leaking seam is a fault; a first-time camper who could not put the tent up is not. `AI_CLASSIFY` reads every Summit Tent return comment since 1 July and gives each one of five labels. It runs in SQL, on every row, and saves the result as a table.

In [ ]:
%%sql -r classify_returns
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS AS
SELECT FEEDBACK_ID, BATCH_CODE, FEEDBACK_DATE,
       AI_CLASSIFY(FEEDBACK_TEXT, ['POLE_FAULT', 'SEAM_FAULT', 'COULD_NOT_PITCH', 'CHANGE_OF_MIND', 'OTHER']):labels[0]::VARCHAR AS LABEL
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE PRODUCT_ID = 'SKU-TNT-0102'
  AND FEEDBACK_SOURCE = 'RETURN_COMMENT'
  AND FEEDBACK_DATE >= '2026-07-01';

Count the labels by batch.

In [ ]:
%%sql -r labels_by_batch
SELECT BATCH_CODE, LABEL, COUNT(*) AS COMMENTS,
       ROUND(RATIO_TO_REPORT(COUNT(*)) OVER (PARTITION BY BATCH_CODE), 3) AS SHARE_OF_BATCH
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

In [ ]:
labels = as_pandas(labels_by_batch, numeric=["SHARE_OF_BATCH"])
order = ["POLE_FAULT", "SEAM_FAULT", "COULD_NOT_PITCH", "CHANGE_OF_MIND", "OTHER"]
colours = {"POLE_FAULT": RED, "SEAM_FAULT": "#E2876F", "COULD_NOT_PITCH": AMBER, "CHANGE_OF_MIND": GREEN, "OTHER": GREY}
shares = labels.pivot(index="BATCH_CODE", columns="LABEL", values="SHARE_OF_BATCH").reindex(columns=order).fillna(0) * 100
fig, ax = plt.subplots(figsize=(8, 2.6))
left = [0] * len(shares)
for label in order:
    ax.barh(shares.index, shares[label], left=left, color=colours[label], label=label.replace("_", " ").lower())
    left = [l + v for l, v in zip(left, shares[label])]
ax.set_xlabel("Share of the batch's return comments (%)")
ax.legend(frameon=False, ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.35))
titled(ax, "B26-07 comes back broken; other batches come back unwanted", "Summit Tent return comments since 1 July, labelled by AI_CLASSIFY")
plt.show()

**What you should see:** seven in ten `B26-07` returns are `POLE_FAULT` or `SEAM_FAULT`. About one in six are `COULD_NOT_PITCH`: first-time campers, not faults. The other batch's returns are mostly `CHANGE_OF_MIND`.

### Faults per tent sold

The labels turn into a rate Kestrel can check: faulty returns as a share of the tents sold from each batch. Only the two batches on sale since 1 July have labelled comments.

In [ ]:
%%sql -r fault_rate
WITH sold AS (
  SELECT BATCH_CODE, SUM(QUANTITY) AS SOLD
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE
  WHERE PRODUCT_ID = 'SKU-TNT-0102' AND BATCH_CODE IS NOT NULL
  GROUP BY 1),
faults AS (
  SELECT BATCH_CODE, COUNT_IF(LABEL IN ('POLE_FAULT', 'SEAM_FAULT')) AS FAULTY_RETURNS
  FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
  GROUP BY 1)
SELECT s.BATCH_CODE, s.SOLD, f.FAULTY_RETURNS, ROUND(f.FAULTY_RETURNS / s.SOLD, 3) AS FAULT_RATE
FROM sold s
JOIN faults f ON f.BATCH_CODE = s.BATCH_CODE
ORDER BY FAULT_RATE DESC;

**What you should see:** about 300 faulty returns from about 3,600 `B26-07` tents sold, roughly 9 in every 100. The other batch has a handful from about 7,000.

### A brief for Kestrel: AI_AGG

Kestrel's quality team will not read 300 comments. `AI_AGG` reads a whole column of text, more than fits in one prompt, and answers one instruction about all of it. Here it writes the fault brief that goes with the claim.

In [ ]:
%%sql -r kestrel_brief
SELECT COUNT(*) AS FAULT_COMMENTS,
       AI_AGG(f.FEEDBACK_TEXT,
         'These are customer return comments about faulty Summit Tents from one production batch. '
         || 'Write a short brief for the supplier''s quality team: what fails, where on the tent it fails, '
         || 'how soon after purchase, and the conditions at the time. Plain Australian English, at most 120 words, no customer names.') AS KESTREL_BRIEF
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS r
JOIN WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK f ON f.FEEDBACK_ID = r.FEEDBACK_ID
WHERE r.BATCH_CODE = 'B26-07'
  AND r.LABEL IN ('POLE_FAULT', 'SEAM_FAULT');

**What you should see:** one paragraph built from about 300 comments: poles cracking or snapping at the hub, seam tape lifting and leaking, usually on the first or second trip, in ordinary conditions. The wording changes from run to run.

## 3. Who has already called about it?

Some Summit Tent owners did not wait to bring the tent back: they called the contact centre. Those customers need a call from their store before their Christmas trip.

`AI_FILTER` answers a yes-or-no question about each row, in plain English, so it can go in a `WHERE` clause or a column. This cell asks it of every tent message to the contact centre: does the customer say their tent broke or leaked and ask for it to be put right?

In [ ]:
%%sql -r tent_calls
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.TENT_CALLS AS
SELECT FEEDBACK_ID, FEEDBACK_DATE, PRODUCT_ID, PRODUCT_NAME, STORE_ID, STATE, FEEDBACK_TEXT,
       AI_FILTER(PROMPT('In this message to an outdoor store, the customer says their tent broke or leaked and asks for a refund, replacement or repair: {0}', FEEDBACK_TEXT)) AS FAULT_CLAIM
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE CATEGORY = 'Tents & Shelters'
  AND FEEDBACK_SOURCE = 'CONTACT_CENTRE';

In [ ]:
%%sql -r calls_summary
SELECT IFF(PRODUCT_ID = 'SKU-TNT-0102', 'Summit Tent', 'Every other tent') AS TENT,
       COUNT(*) AS MESSAGES,
       COUNT_IF(FAULT_CLAIM) AS FAULT_CLAIMS,
       ROUND(FAULT_CLAIMS / MESSAGES, 2) AS SHARE
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_CALLS
GROUP BY 1
ORDER BY 1 DESC;

**What you should see:** about nine in ten Summit Tent messages are a fault claim, against about one in seven for every other tent.

The call-back list for the stores, newest first.

In [ ]:
%%sql -r call_backs
SELECT c.FEEDBACK_DATE, s.STORE_NAME, c.STATE, LEFT(c.FEEDBACK_TEXT, 120) || '...' AS MESSAGE
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_CALLS c
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = c.STORE_ID
WHERE c.PRODUCT_ID = 'SKU-TNT-0102'
  AND c.FAULT_CLAIM
ORDER BY c.FEEDBACK_DATE DESC;

**What you should see:** about two dozen customers, in stores across the country, most writing about a pole snapping at the hub or seams leaking on a family trip.

## 4. Who gets a sleeping-bag offer?

Every family with a new tent needs somewhere to sleep, and some of this week's tent buyers left without a sleeping bag. The Basecamp members among them can be sent an offer.

### Find the Basecamp members

Members who bought a tent from 1 to 3 November with no sleeping bag or mat in the same order, one row each, with their first name, store and tent.

In [ ]:
%%sql -r sleeping_bag_offer
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER AS
WITH tent_orders AS (
  SELECT f.ORDER_ID, f.ORDER_DATE, f.MEMBER_ID, f.STORE_ID,
         MAX(IFF(p.CATEGORY = 'Tents & Shelters', p.PRODUCT_NAME, NULL)) AS TENT_NAME,
         MAX(IFF(p.KIT_GROUP = 'SLEEP', 1, 0)) AS HAS_SLEEP
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03' AND f.MEMBER_ID IS NOT NULL
  GROUP BY 1, 2, 3, 4)
SELECT m.MEMBER_ID, m.FIRST_NAME, s.STORE_NAME, t.TENT_NAME, NULL::VARCHAR AS EMAIL
FROM tent_orders t
JOIN WILDERNESS_HOL.RAW.DIM_MEMBER m ON m.MEMBER_ID = t.MEMBER_ID
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = t.STORE_ID
WHERE t.TENT_NAME IS NOT NULL AND t.HAS_SLEEP = 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY m.MEMBER_ID ORDER BY t.ORDER_DATE, t.ORDER_ID) = 1;

**What you should see:** about 700 members in `SLEEPING_BAG_OFFER`.

### Price the run before you write 700 emails

`AI_COMPLETE` is charged for the prompt it reads and the email it writes. This counts the prompt tokens for every member against the five written below. The emails it writes come on top, so treat the count as the least the run costs.

In [ ]:
%%sql -r email_cost
WITH prompts AS (
  SELECT 'Write a short email from Wilderness Outfitters, an Australian outdoor retailer, to a Basecamp loyalty member. '
         || 'They are ' || FIRST_NAME || ', who bought a ' || TENT_NAME || ' at ' || STORE_NAME
         || ' this week but no sleeping bag or mat. Thank them for their tent, and offer 15% off a sleeping bag or mat '
         || 'before Christmas. Plain Australian English, under 80 words, no exclamation marks, no emoji. '
         || 'Return only a subject line and the body.' AS PROMPT_TEXT
  FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER)
SELECT COUNT(*) AS MEMBERS,
       SUM(AI_COUNT_TOKENS('ai_complete', 'claude-sonnet-4-6', PROMPT_TEXT)) AS PROMPT_TOKENS_ALL,
       ROUND(PROMPT_TOKENS_ALL / MEMBERS) AS PROMPT_TOKENS_EACH,
       5 * PROMPT_TOKENS_EACH AS PROMPT_TOKENS_FOR_FIVE
FROM prompts;

**What you should see:** about 700 members at about 110 prompt tokens each: about 74,000 for everyone, about 550 for the five.

### Write five offer emails: AI_COMPLETE

`AI_COMPLETE` writes text in SQL. Each member's name, tent and store go into the prompt, and the email comes back in the `EMAIL` column. It writes five for review before anyone writes 700. Nothing here sends an email.

In [ ]:
%%sql -r offer_emails
UPDATE WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER
SET EMAIL = AI_COMPLETE('claude-sonnet-4-6',
  'Write a short email from Wilderness Outfitters, an Australian outdoor retailer, to a Basecamp loyalty member. '
  || 'They are ' || FIRST_NAME || ', who bought a ' || TENT_NAME || ' at ' || STORE_NAME
  || ' this week but no sleeping bag or mat. Thank them for their tent, and offer 15% off a sleeping bag or mat '
  || 'before Christmas. Plain Australian English, under 80 words, no exclamation marks, no emoji. '
  || 'Return only a subject line and the body.')::VARCHAR
WHERE MEMBER_ID IN (SELECT MEMBER_ID FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER ORDER BY MEMBER_ID LIMIT 5);

In [ ]:
%%sql -r show_emails
SELECT FIRST_NAME, TENT_NAME, STORE_NAME, EMAIL
FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER
WHERE EMAIL IS NOT NULL
ORDER BY MEMBER_ID;

**What you should see:** five short emails, each naming the member, their tent and their store. The wording changes from run to run.

### Check the drafts before anyone sends them

A reviewer checks the length and the tone by eye. SQL can check them on every draft: the word count, no exclamation marks, and, with `AI_FILTER`, that the email makes the one offer and promises nothing else.

In [ ]:
%%sql -r email_check
SELECT FIRST_NAME, TENT_NAME,
       REGEXP_COUNT(EMAIL, '\\S+') AS WORDS,
       CONTAINS(EMAIL, '!') AS HAS_EXCLAMATION,
       AI_FILTER(PROMPT('This email from a retailer to a customer makes exactly one offer, 15% off a sleeping bag or mat, and promises nothing else: {0}', EMAIL)) AS ONE_OFFER_ONLY
FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER
WHERE EMAIL IS NOT NULL
ORDER BY MEMBER_ID;

**What you should see:** every draft under 80 words, `HAS_EXCLAMATION` false and `ONE_OFFER_ONLY` true. A draft that fails goes back for rewriting, not out to a member.

## 5. What does each store do?

Part 2 showed the faulty batch is real, so every unsold `B26-07` Summit Tent goes back to Kestrel. Now put it all together, store by store: the good tents each store has, what it will sell by Christmas Eve, and whether it has tents to spare or needs more.

### Build the plan as a dynamic table

One row per store. The faulty `B26-07` tents come off first. The forecast to Christmas Eve is last year's tent sales from 4 November to 24 December, grown by the state's growth so far. A store with more good tents than its forecast moves the spare out (`MOVE OUT`); a store with fewer needs tents moved in (`MOVE IN`). Snowflake refreshes the table within an hour of new data, with no job to schedule.

In [ ]:
%%sql -r christmas_plan
CREATE OR REPLACE DYNAMIC TABLE WILDERNESS_HOL.LAB2_ANALYTICS.CHRISTMAS_PLAN
  TARGET_LAG = '1 hour'
  WAREHOUSE = WILDERNESS_WH
  COMMENT = 'Christmas plan at 3 Nov 2026: one row per store, tents to send back and move'
AS
WITH tents AS (
  SELECT PRODUCT_ID FROM WILDERNESS_HOL.RAW.DIM_PRODUCT WHERE CATEGORY = 'Tents & Shelters'),
growth AS (
  SELECT s.STATE,
         SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0))
         / NULLIF(SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)), 0) AS GROWTH
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03'
  GROUP BY 1),
last_year AS (
  SELECT f.STORE_ID, SUM(f.QUANTITY) AS TENTS_SOLD_LY
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-04' AND '2025-12-24'
  GROUP BY 1),
stock AS (
  SELECT i.STORE_ID, SUM(i.ON_HAND_UNITS) AS TENTS_ON_HAND,
         SUM(IFF(i.BATCH_CODE = 'B26-07', i.ON_HAND_UNITS, 0)) AS TENTS_TO_SEND_BACK
  FROM WILDERNESS_HOL.RAW.FCT_INVENTORY i
  JOIN tents t ON t.PRODUCT_ID = i.PRODUCT_ID
  GROUP BY 1),
plan AS (
  SELECT s.STORE_ID, s.STORE_NAME, s.STATE, s.LATITUDE, s.LONGITUDE,
         st.TENTS_ON_HAND, st.TENTS_TO_SEND_BACK,
         ROUND(ly.TENTS_SOLD_LY * g.GROWTH) AS FORECAST_TO_XMAS
  FROM WILDERNESS_HOL.RAW.DIM_STORE s
  JOIN stock st ON st.STORE_ID = s.STORE_ID
  JOIN last_year ly ON ly.STORE_ID = s.STORE_ID
  JOIN growth g ON g.STATE = s.STATE)
SELECT STORE_ID, STORE_NAME, STATE, LATITUDE, LONGITUDE, TENTS_ON_HAND, TENTS_TO_SEND_BACK, FORECAST_TO_XMAS,
       TENTS_ON_HAND - TENTS_TO_SEND_BACK - FORECAST_TO_XMAS AS GAP,
       IFF(TENTS_ON_HAND - TENTS_TO_SEND_BACK - FORECAST_TO_XMAS > 0, 'MOVE OUT', 'MOVE IN') AS ACTION
FROM plan;

### The plan in four numbers

In [ ]:
%%sql -r plan_totals
SELECT SUM(TENTS_TO_SEND_BACK) AS TENTS_TO_SEND_BACK,
       SUM(IFF(ACTION = 'MOVE OUT', GAP, 0)) AS TENTS_SPARE,
       -SUM(IFF(ACTION = 'MOVE IN', GAP, 0)) AS TENTS_SHORT,
       TENTS_SHORT - TENTS_SPARE AS TENTS_TO_REORDER,
       COUNT_IF(ACTION = 'MOVE OUT') AS STORES_MOVE_OUT,
       COUNT_IF(ACTION = 'MOVE IN') AS STORES_MOVE_IN
FROM WILDERNESS_HOL.LAB2_ANALYTICS.CHRISTMAS_PLAN;

**What you should see:** about 5,100 tents to send back, about 2,400 spare and about 5,300 short. The spare does not cover the shortfall, so about 2,900 tents are reordered by 13 November. 39 stores move tents out and 33 move them in.

### The plan by state

In [ ]:
%%sql -r plan_by_state
SELECT STATE, COUNT(*) AS STORES, SUM(TENTS_TO_SEND_BACK) AS TENTS_TO_SEND_BACK, SUM(GAP) AS GAP
FROM WILDERNESS_HOL.LAB2_ANALYTICS.CHRISTMAS_PLAN
GROUP BY 1
ORDER BY GAP;

In [ ]:
by_state = as_pandas(plan_by_state, numeric=["GAP"]).sort_values("GAP")
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.barh(by_state["STATE"], by_state["GAP"], color=[RED if g < 0 else GREEN for g in by_state["GAP"]])
ax.bar_label(bars, labels=[f"{g:+,.0f}" for g in by_state["GAP"]], padding=4, color=INK)
ax.set_xlim(by_state["GAP"].min() * 1.3, by_state["GAP"].max() * 1.3)
ax.axvline(0, color=INK, linewidth=1)
ax.set_xlabel("Good tents minus the forecast to 24 Dec")
titled(ax, "Move tents north, then reorder the rest", "Green: tents to spare. Red: tents short, after the B26-07 tents go back")
plt.show()

**What you should see:** New South Wales and Queensland short, every other state with tents to spare.

## What the plan tells Martha

- **Send back:** about 5,100 `B26-07` Summit Tents to Kestrel, with the fault rate and the brief from Part 2 as the evidence.
- **Move:** about 2,400 spare tents from the other states to New South Wales and Queensland.
- **Reorder:** about 2,900 tents for New South Wales and Queensland, by 13 November.
- **Call back:** the customers on the list in Part 3, before their Christmas trips.
- **Offer:** about 700 Basecamp members get the sleeping-bag offer once the five drafts pass review.

## What is next

The merchandising team opens the plan in the `CHRISTMAS_PLANNER` app: in Snowsight, go to **Projects » Streamlit**. If you did not build it in the lab, the catch-up script `91_catch_up_lab2.sql` deploys the finished one. The tables this notebook built stay in `LAB2_ANALYTICS` for the app and for Lab 3.